In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [17]:
#Import Necessary Libraries
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

In [9]:
train_transform = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [10]:
from torchvision import datasets
train_dataset = torchvision.datasets.MNIST(root = './data' ,transform = train_transform, train = True , download = True)
val_dataset = torchvision.datasets.MNIST(root = './data' ,transform = val_transform, train = False , download = True)
print("DataSet Downloaded...")

DataSet Downloaded...


In [27]:
num_classes = 10
batch_size = 64
num_channels = 1
patch_size = 7
img_size = 28
num_patches = (img_size // patch_size)**2
embedding_dim = 64
attention_heads = 4
transformer_blocks = 4
learning_rate = 0.001
epochs = 5
mlp_hidden_nodes = 128

In [12]:
train_loader = DataLoader(train_dataset,batch_size = batch_size , shuffle = True)
val_loader = DataLoader(val_dataset,batch_size = batch_size , shuffle = False)

In [48]:
#patch_embedding
class Patch_Embedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embed = nn.Conv2d(num_channels,embedding_dim,kernel_size = patch_size , stride = patch_size)

    def forward(self,x):
        x = self.patch_embed(x)
        x = x.flatten(2)
        x = x.transpose(1,2)
        return x

In [49]:
class TransformerEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.layer_norm2 = nn.LayerNorm(embedding_dim)
        self.multi_head_attention = nn.MultiheadAttention(embedding_dim,attention_heads,batch_first = True)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim,mlp_hidden_nodes),
            nn.GELU(),
            nn.Linear(mlp_hidden_nodes,embedding_dim)
        )

    def forward(self,x):
        residual_1 = x
        x = self.layer_norm1(x)
        x = self.multi_head_attention(x,x,x)[0]
        x = x + residual_1

        residual_2 = x
        x = self.layer_norm2(x)
        x = self.mlp(x)
        x = x + residual_2
        
        return x

In [50]:
class MLP_head(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.mlp_head = nn.Linear(embedding_dim,num_classes)

    def forward(self,x):
        x = self.layer_norm1(x)
        x = self.mlp_head(x)

        return x 

In [51]:
class VisionTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embedding = Patch_Embedding()
        self.cls_token = nn.Parameter(torch.randn(1,1,embedding_dim))
        self.position_embedding = nn.Parameter(torch.randn(1,1 + num_patches , embedding_dim))
        self.transformer_blocks = nn.Sequential(*[TransformerEncoder() for _ in range(transformer_blocks)])
        self.mlp_head = MLP_head()

    def forward(self,x):
        x = self.patch_embedding(x)
        B = x.size(0)
        class_token = self.cls_token.expand(B,-1,-1)
        x = torch.cat((class_token,x),dim = 1)
        x = x + self.position_embedding
        x = self.transformer_blocks(x)
        x = x[:,0]
        x = self.mlp_head(x)

        return x
        

In [32]:
print(embedding_dim, type(embedding_dim))
print(num_classes, type(num_classes))

64 <class 'int'>
10 <class 'int'>


In [54]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = VisionTransformer().to(device)
optimizer = torch.optim.Adam(model.parameters(),lr = learning_rate)
criterion = nn.CrossEntropyLoss()

In [59]:
for epoch in range(epochs):

    model.train()

    total_loss = 0
    correct_epoch = 0
    total_epoch = 0

    print(f"\nEpoch {epoch + 1}")

    for batch_idx, (images, labels) in enumerate(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct = (labels == predictions).sum().item()
        correct_epoch += correct

        total_epoch += labels.size(0)

    accuracy = 100 * correct_epoch / total_epoch
    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}: "
        f"Accuracy = {accuracy:.2f}% | "
        f"Loss = {average_loss:.4f}"
    )


Epoch 1
Epoch 1: Accuracy = 96.76% | Loss = 0.1042

Epoch 2
Epoch 2: Accuracy = 97.17% | Loss = 0.0902

Epoch 3
Epoch 3: Accuracy = 97.46% | Loss = 0.0793

Epoch 4
Epoch 4: Accuracy = 97.71% | Loss = 0.0727

Epoch 5
Epoch 5: Accuracy = 97.82% | Loss = 0.0682


In [63]:
model.eval()
total = 0
total_correct = 0
total_loss = 0
with torch.no_grad():
    for (images,labels) in val_loader:
        images , labels = images.to(device) , labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs,labels)
        preds = outputs.argmax(dim = 1)
        correct = (preds == labels).sum().item()
        total_loss += loss.item()
        total += labels.size(0)
        total_correct += correct

    accuracy = 100 * total_correct / total
    print(f"The Eval Accuracy is {accuracy}")
    print(f"The Eval loss is {total_loss/len(val_loader)}")
        

The Eval Accuracy is 98.02
The Eval loss is 0.06304014921555397
